## מה לא לשים ב-Git: `.gitignore`

Git נועד לעקוב אחרי **קוד** — קבצים קטנים, טקסטואליים, שהשינויים בהם משמעותיים. הוא **לא** מתאים למעקב אחרי כל קובץ בתיקייה:

- **קבצים שנוצרים אוטומטית** — `__pycache__/`, `.ipynb_checkpoints/` — אפשר לשחזר אותם בכל רגע מהקוד עצמו.
- **נתוני מדידה גולמיים וקבצי פלט גדולים** — קבצי CSV של כמה עשרות מגה-בייט, תמונות, גרפים שנשמרו. אלה תופסים מקום עצום בהיסטוריה (שנשארת שם **לצמיתות**, גם אם תמחקו את הקובץ בקומיט הבא), ולרוב עדיף שיישמרו בנפרד (דיסק משותף, שרת מדידות) ולא ב-Git.
- **קבצים אישיים למחשב שלכם** — כמו `.DS_Store` של macOS.

הפתרון: קובץ בשם **`.gitignore`** בשורש הריפו, עם רשימת תבניות קבצים ל"התעלמות". קובץ שתואם לתבנית שם פשוט לא יופיע ב-`git status` כקובץ לא-עוקב, ו-`git add .` לא יכלול אותו.

```{note}
`.gitignore` משפיע רק על קבצים **שעדיין לא עוקבים אחריהם**. אם קובץ כבר נמצא בהיסטוריה (כבר עשיתם לו `git add`+`commit` בעבר), הוספתו ל-`.gitignore` לא תפסיק את המעקב אחריו — לכך צריך פקודה נפרדת (`git rm --cached <קובץ>`). הכי בטוח: להוסיף ל-`.gitignore` **לפני** ה-commit הראשון של קבצים כאלה.
```

### דוגמה: תיקיית עבודה מבולגנת

נדמה תיקיית פרויקט טיפוסית של פיזיקאי: קוד ניתוח, וגם קבצים שלא היינו רוצים ב-Git — קאש של פייתון, checkpoints של Jupyter, ותיקיית נתוני מדידה גולמיים.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_gitignore_")
os.chdir(workdir)

!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

os.makedirs("__pycache__", exist_ok=True)
os.makedirs(".ipynb_checkpoints", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

with open("analysis.py", "w") as f:
    f.write("print('analysis')\n")
open("__pycache__/analysis.cpython-310.pyc", "w").close()
open(".ipynb_checkpoints/analysis-checkpoint.ipynb", "w").close()
open("data/raw/measurement_run_001.csv", "w").close()

!git status --short

`??` בפלט `git status --short` מסמן קבצים לא-עוקבים. ארבע שורות — אבל רק `analysis.py` הוא קוד שבאמת רוצים לעקוב אחריו. נכתוב `.gitignore`:

In [ ]:
%%writefile .gitignore
__pycache__/
.ipynb_checkpoints/
data/raw/

In [ ]:
!git status --short

עכשיו נשארו רק `.gitignore` עצמו (שאותו כן רוצים לעקוב אחריו — הוא חלק מהפרויקט) ו-`analysis.py`. ה"רעש" נעלם.

### הערה טכנית: Git ומחברות Jupyter

קובץ `.ipynb` הוא בעצם JSON, שכולל בתוכו גם את **הפלטים** (טקסט, גרפים) של כל תא. המשמעות: `git diff` בין שתי גרסאות של מחברת נראה כמו רעש כמעט בלתי-קריא — כל שינוי קטן בפלט (למשל timestamp) מציג diff ענק, גם אם הקוד עצמו לא השתנה כמעט.

שתי דרכים מקובלות להתמודד עם זה:
1. **לנקות פלטים לפני commit** (Kernel → Restart & Clear Output ב-Jupyter) — כך ה-diff נשאר קריא, אבל מאבדים את הפלטים השמורים.
2. **כלים ייעודיים** כמו `nbstripout` (מנקה פלטים אוטומטית בכל commit) או `nbdime` (מציג diff חכם שמבין את מבנה ה-JSON). לא נלמד אותם לעומק השבוע, אבל שווה להכיר את השמות אם תרצו לחפור בעתיד.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "קובץ כבר נמצא בהיסטוריית Git (כבר עשיתם לו add+commit). אם עכשיו תוסיפו אותו ל-.gitignore, מה יקרה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "הוא יימחק אוטומטית מההיסטוריה של הריפו", "correct": False, "feedback": "לא — .gitignore לעולם לא מוחק דבר מההיסטוריה."},
            {"answer": "שום דבר לא ישתנה — Git ימשיך לעקוב אחריו כרגיל, .gitignore חל רק על קבצים לא-עוקבים", "correct": True, "feedback": "נכון — זו בדיוק ההערה החשובה למעלה."},
            {"answer": "Git יתריע בשגיאה בפעם הבאה שתעשו commit", "correct": False, "feedback": "לא, אין שגיאה — פשוט הקובץ ממשיך להיות עוקב כרגיל, בלי שום התראה."},
            {"answer": "השינויים הבאים בו כבר לא יישמרו בקומיטים", "correct": False, "feedback": "לא נכון — עדיין אפשר (ואפשר גם בטעות) לעשות לו git add ולשמור בו שינויים."}
        ]
    },
    {
        "question": "למה קובצי נתונים גולמיים גדולים (CSV של עשרות MB) הם בעייתיים במיוחד ב-Git, יותר מאשר בכל אמצעי אחסון אחר?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "Git לא מסוגל טכנית לשמור קבצים גדולים בכלל", "correct": False, "feedback": "טכנית הוא כן יכול, הבעיה היא תפעולית לא טכנית."},
            {"answer": "כל גרסה של הקובץ נשארת בהיסטוריה לצמיתות, כך שגודל הריפו רק תופח עם הזמן ולעולם לא מצטמצם", "correct": True, "feedback": "נכון."},
            {"answer": "GitHub חוסם העלאת קבצי CSV לגמרי", "correct": False, "feedback": "לא נכון — GitHub לא חוסם CSV; הבעיה היא בגודל ההיסטוריה המצטבר, לא בסוג הקובץ."},
            {"answer": "אין שום בעיה אמיתית, זו רק המלצת סגנון", "correct": False, "feedback": "יש בעיה תפעולית ממשית: ריפו שמנפח לגיגה-בייטים הופך איטי לשכפול ולעבודה יומיומית."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

בתיקייה הבאה (נבנתה למטה) יש קוד ניתוח, וגם: קבצי גרפים שנשמרו (`*.png`), תיקיית תוצאות זמניות (`results/`), וקאש של פייתון. כתבו `.gitignore` כך שאחרי כתיבתו, `git status --short` יראה **רק** את `experiment.py` ואת `.gitignore` עצמו.

In [ ]:
workdir2 = tempfile.mkdtemp(prefix="git_gitignore_practice_")
os.chdir(workdir2)
!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

os.makedirs("__pycache__", exist_ok=True)
os.makedirs("results", exist_ok=True)
with open("experiment.py", "w") as f:
    f.write("print('experiment')\n")
open("__pycache__/experiment.cpython-310.pyc", "w").close()
open("results/summary.txt", "w").close()
open("fit_plot.png", "w").close()

!git status --short

In [ ]:
# כתבו כאן את תוכן ה-.gitignore, ואז בדקו עם git status --short

`````{admonition} פתרון
:class: dropdown, tip
```python
%%writefile .gitignore
__pycache__/
results/
*.png
```
```python
!git status --short
```
תוצאה: רק `?? .gitignore` ו-`?? experiment.py`.
`````